# 디지털트윈 자동 분류 — 불량 / 정상파랑 / 정상초록
Factory I/O 장면 **Vision_Color_3Way**(과제용 수정본)를 열고, 드라이버 **Modbus TCP/IP Server**를 START 한 뒤 셀을 위에서부터 실행하세요. 커널: **Python (cherry venv)**

| 경로 | 판별 | 동작 |
|---|---|---|
| 불량 (가공 전 원자재: 파랑=1, 초록=4) | 비전1 | 푸셔0 → 측면 분기 1 |
| 정상 파랑 (제품 베이스=3, 리드=2) | 비전2 | 푸셔1 → 측면 분기 2 |
| 정상 초록 (제품 베이스=6, 리드=5) | 비전2 | 직진 배출 |

### 모드버스 I/O 맵 (서버 192.168.0.16:502, Slave ID 1)
| 구분 | 주소 | 태그 | 의미 |
|---|---|---|---|
| 입력(DI) | 0 / 1 | Pusher 0 Back / Front Limit | 푸셔0 후진/전진 완료 |
| 입력(DI) | 2 / 3 | FACTORY I/O Running / Reset | 시뮬레이션 상태 |
| 입력(DI) | 4 / 5 | Pusher 1 Back / Front Limit | 푸셔1 후진/전진 완료 |
| 입력(DI) | 6 / 7 | Start / Stop Button | 제어반 (Stop은 NC 접점) |
| 입력레지스터 | 0 / 1 | Vision Sensor 0 / 1 (Value) | 비전1 / 비전2 판별값 |
| 코일 | 0 | Belt Conveyor 6m | 메인 컨베이어 |
| 코일 | 1 / 6 / 5 | Belt 4m 0 / 4m 1 / 4m 2 | 불량 / 파랑 / 초록(직진) 배출 |
| 코일 | 2 | Emitter 0 | 투입 |
| 코일 | 3 / 4 | Pusher 0 / 1 | 푸셔 (모노스테이블) |
| 코일 | 7 / 8 / 9 | Remover 0 / 1 / 2 | 경로별 제거 |
| 코일 | 10 / 11 | Start / Stop Light | 제어반 램프 |
| 코일 | 12 | FACTORY I/O Run | 시뮬레이션 실행 |
| 홀딩레지스터 | 0 / 1 / 2 | Digital Display 0 / 1 / 2 | 불량 / 파랑 / 초록 수량 |


In [1]:
"""Factory I/O 3-way sorting: defect / normal blue / normal green."""
import time, csv
from collections import Counter
from pymodbus.client import ModbusTcpClient

HOST, DEVICE_ID = '192.168.0.16', 1
DEFECT = {1, 4}                      # 가공 전 원자재(파랑, 초록)
BLUE, GREEN = {2, 3}, {5, 6}         # 정상 파랑 / 정상 초록 (리드, 베이스)
# coil addresses
C_MAIN, C_BAD, C_EMIT, C_P0, C_P1, C_GREEN, C_BLUE = 0, 1, 2, 3, 4, 5, 6
C_REMOVERS, C_START_L, C_STOP_L, C_RUN = (7, 8, 9), 10, 11, 12
FEED_COILS = (C_MAIN, C_EMIT)                     # 품목 투입/이동
EXIT_COILS = (C_BAD, C_GREEN, C_BLUE) + C_REMOVERS  # 배출 경로는 계속 동작
# discrete inputs
I_P0_BACK, I_P0_FRONT, I_RUNNING, I_RESET, I_P1_BACK, I_P1_FRONT, I_START, I_STOP = range(8)
BACK = {0: I_P0_BACK, 1: I_P1_BACK}
FRONT = {0: I_P0_FRONT, 1: I_P1_FRONT}

def checked(r):
    if r.isError():
        raise RuntimeError(str(r))
    return r

class Sorter:
    def __init__(self, host=HOST):
        self.client = ModbusTcpClient(host, port=502, timeout=2, retries=0)
        self.counts = Counter()
        self.events = []
        self.stop_requested = False

    # ---- I/O helpers
    def write(self, addr, value):
        checked(self.client.write_coil(addr, bool(value), device_id=DEVICE_ID))
    def feed(self, on):
        for a in FEED_COILS: self.write(a, on)
    def show(self):
        checked(self.client.write_registers(0, [self.counts['DEFECT'], self.counts['BLUE'], self.counts['GREEN']], device_id=DEVICE_ID))
    def event(self, text):
        line = time.strftime('%H:%M:%S ') + text
        self.events.append(line); print(line, flush=True)
    def connect(self):
        if not self.client.connect():
            self.client.close()
            raise ConnectionError(f'{HOST}:502 연결 실패 — Factory I/O 드라이버 화면에서 START를 누르세요.')

    def run(self, duration=120, plan=None):
        """duration초 동안 연속 운전. plan=[(t초,'stop'|'start'), ...] 이면 코드로 정지/재시작 시험."""
        plan = sorted(plan or [])
        self.connect()
        self.stop_requested = False
        stations = [dict(cand=0, since=0.0, armed=False, pending=None) for _ in range(2)]
        state, active, moving, halted, halt_req = 'START', None, False, False, False
        t0 = since = prev = time.monotonic()
        prev_start = prev_stop = None
        try:
            checked(self.client.write_coils(0, [False]*12 + [True], device_id=DEVICE_ID))  # Run ON
            self.show()
            self.event('START  불량→P0 / 파랑→P1 / 초록→직진')
            while time.monotonic() - t0 < duration and not self.stop_requested:
                now = time.monotonic(); dt, prev = now - prev, now
                bits = checked(self.client.read_discrete_inputs(0, count=8, device_id=DEVICE_ID)).bits
                # ---- 정지/재시작 요청 (제어반 버튼: Start 상승엣지, Stop(NC) 하강엣지 / 코드 plan)
                start_edge = prev_start is not None and bits[I_START] and not prev_start
                stop_edge = prev_stop is not None and prev_stop and not bits[I_STOP]
                prev_start, prev_stop = bits[I_START], bits[I_STOP]
                while plan and now - t0 >= plan[0][0]:
                    cmd = plan.pop(0)[1]
                    stop_edge |= cmd == 'stop'; start_edge |= cmd == 'start'
                if stop_edge and not halted:
                    halt_req = True; self.event('STOP 요청 — 현재 동작 완료 후 정지')
                if start_edge and halted:
                    halted, halt_req = False, False
                    self.write(C_STOP_L, False); self.write(C_START_L, True)
                    self.feed(True); moving = True
                    self.event('RESTART'); self.counts['RESTART'] += 1
                if state == 'START':
                    if bits[I_RUNNING] and bits[I_P0_BACK] and bits[I_P1_BACK]:
                        for a in EXIT_COILS + (C_START_L,): self.write(a, True)
                        self.feed(True)
                        state, moving, since = 'FEED', True, now
                    elif now - since > 10:
                        miss = [n for n, i in (('Running', I_RUNNING), ('P0 Back', I_P0_BACK), ('P1 Back', I_P1_BACK)) if not bits[i]]
                        raise TimeoutError('시작 신호 없음: ' + ', '.join(miss))
                    time.sleep(.02); continue
                if not bits[I_RUNNING] or bits[I_RESET]:
                    raise RuntimeError('Factory I/O 정지 또는 리셋됨')
                # ---- 비전 판별 (값이 0으로 돌아온 뒤에만 다음 품목으로 인정)
                codes = checked(self.client.read_input_registers(0, count=2, device_id=DEVICE_ID)).registers
                for i, (s, code) in enumerate(zip(stations, codes)):
                    if code != s['cand']: s['cand'], s['since'] = code, now
                    if now - s['since'] >= .04:
                        if code == 0:
                            s['armed'] = True
                        elif s['armed'] and s['pending'] is None:
                            s['armed'] = False
                            self.event(f'VISION{i+1} 값={code}')
                            if i == 0 and code in DEFECT:
                                s['pending'] = {'code': code, 'travel': 0.0}      # 판별값 유지
                            elif i == 1 and code in BLUE:
                                s['pending'] = {'code': code, 'travel': 0.0}
                            elif i == 1 and code in GREEN:
                                self.counts['GREEN'] += 1; self.show(); self.event('정상초록 → 직진 배출')
                            elif i == 1 and code in DEFECT:
                                raise RuntimeError('불량품이 P0를 통과함 — 위치 확인 필요')
                            elif code not in DEFECT | BLUE | GREEN:
                                raise RuntimeError(f'알 수 없는 비전값 {code}')
                    if moving and s['pending'] is not None:
                        s['pending']['travel'] += dt
                # ---- 상태기계: 한 번에 하나의 푸셔만, 복귀 리밋 확인 후 다음 투입
                if state == 'FEED':
                    ready = [i for i in (1, 0) if stations[i]['pending'] and stations[i]['pending']['travel'] >= .20]
                    if ready:
                        active = ready[0]; self.feed(False); moving = False
                        state, since = 'SETTLE', now
                    elif halt_req:
                        self.feed(False); moving = False; halted, halt_req = True, False
                        self.write(C_START_L, False); self.write(C_STOP_L, True)
                        self.event('STOPPED (Start 버튼 또는 plan으로 재시작)'); self.counts['STOP'] += 1
                elif state == 'SETTLE' and now - since >= .10:
                    if not bits[BACK[active]]: raise RuntimeError(f'P{active} 후진 리밋 아님')
                    self.write(C_P0 + active, True); state, since = 'EXTEND', now      # 전진 명령
                elif state == 'EXTEND':
                    if bits[FRONT[active]]:
                        self.write(C_P0 + active, False); state, since = 'RETRACT', now  # 전진 완료 → 후진 명령
                    elif now - since > 3: raise TimeoutError(f'P{active} 전진 시간초과')
                elif state == 'RETRACT':
                    if bits[BACK[active]] and not bits[FRONT[active]]:
                        color = 'DEFECT' if active == 0 else 'BLUE'
                        self.counts[color] += 1; self.show()
                        self.event(('불량' if active == 0 else '정상파랑') + f' P{active} 복귀 확인')
                        stations[active]['pending'] = None; stations[active]['armed'] = False
                        state, since = 'CLEAR', now
                        if not halted:
                            self.feed(True); moving = True
                    elif now - since > 3: raise TimeoutError(f'P{active} 후진 시간초과')
                elif state == 'CLEAR':
                    if codes[active] == 0: state = 'FEED'
                    elif now - since > 3: raise RuntimeError(f'P{active} 비전값이 0으로 돌아오지 않음')
                time.sleep(.02)
        except KeyboardInterrupt:
            self.event('사용자 중단')
        finally:
            try:
                checked(self.client.write_coils(0, [False]*12, device_id=DEVICE_ID))
                self.event(f'END {dict(self.counts)} state={state}')
            finally:
                self.client.close()
        return {k: self.counts[k] for k in ('DEFECT', 'BLUE', 'GREEN', 'STOP', 'RESTART')}


In [2]:
# 실행 전 점검 (읽기 전용)
def check_connection():
    c = ModbusTcpClient(HOST, port=502, timeout=2, retries=0)
    if not c.connect():
        print(f'✗ {HOST}:502 연결 실패 → Factory I/O 드라이버에서 START'); return False
    try:
        b = checked(c.read_discrete_inputs(0, count=8, device_id=DEVICE_ID)).bits[:8]
        v = checked(c.read_input_registers(0, count=2, device_id=DEVICE_ID)).registers
    finally:
        c.close()
    names = ['P0 Back','P0 Front','Running','Reset','P1 Back','P1 Front','Start','Stop(NC)']
    print('✓ 연결 OK'); print({n: int(x) for n, x in zip(names, b)}); print('비전값:', v)
    return True
check_connection()


✓ 연결 OK
{'P0 Back': 1, 'P0 Front': 0, 'Running': 1, 'Reset': 0, 'P1 Back': 1, 'P1 Front': 0, 'Start': 0, 'Stop(NC)': 1}
비전값: [0, 0]


True

In [3]:
# 연속 운전 (Jupyter ■ 중지 또는 제어반 Stop 버튼 → 현재 동작 완료 후 정지, Start 버튼 → 재시작)
sorter = Sorter()
result = sorter.run(duration=120)
print('분류 결과:', result)


14:08:18 START  불량→P0 / 파랑→P1 / 초록→직진
14:08:22 VISION1 값=1
14:08:24 불량 P0 복귀 확인
14:08:28 VISION1 값=4
14:08:30 불량 P0 복귀 확인
14:08:34 VISION1 값=1
14:08:36 불량 P0 복귀 확인
14:08:40 VISION1 값=3
14:08:42 VISION2 값=3
14:08:44 정상파랑 P1 복귀 확인
14:08:48 VISION1 값=1
14:08:50 불량 P0 복귀 확인
14:08:54 VISION1 값=1
14:08:56 불량 P0 복귀 확인
14:09:00 VISION1 값=6
14:09:02 VISION2 값=6
14:09:02 정상초록 → 직진 배출
14:09:07 VISION1 값=1
14:09:09 불량 P0 복귀 확인
14:09:13 VISION1 값=6
14:09:15 VISION2 값=6
14:09:15 정상초록 → 직진 배출
14:09:20 VISION1 값=3
14:09:22 VISION2 값=3
14:09:25 정상파랑 P1 복귀 확인
14:09:27 VISION1 값=4
14:09:30 불량 P0 복귀 확인
14:09:33 VISION1 값=4
14:09:36 불량 P0 복귀 확인
14:09:39 VISION1 값=3
14:09:42 VISION2 값=3
14:09:44 정상파랑 P1 복귀 확인
14:09:48 VISION1 값=4
14:09:50 불량 P0 복귀 확인
14:09:54 VISION1 값=6
14:09:56 VISION2 값=6
14:09:56 정상초록 → 직진 배출
14:09:59 VISION1 값=1
14:10:02 불량 P0 복귀 확인
14:10:05 VISION1 값=4
14:10:08 불량 P0 복귀 확인
14:10:11 VISION1 값=4
14:10:14 불량 P0 복귀 확인
14:10:17 VISION1 값=1
14:10:18 END {'DEFECT': 12, 'BLUE': 3, 'GREEN': 3}

## 검증 (코드로 정지·재시작 시험, 추가점수)
4종 혼합 투입 상태에서 60초 운전 → 20초 시점 정지 → 30초 시점 재시작. 오류 없이 끝나고 정지 중 분류가 늘지 않으면 합격입니다. 결과는 `검증_결과.csv`에 기록됩니다.

In [4]:
def verify(duration=60, stop_at=20, restart_at=30):
    s = Sorter()
    error = None
    try:
        res = s.run(duration=duration, plan=[(stop_at, 'stop'), (restart_at, 'start')])
    except Exception as e:
        error = f'{type(e).__name__}: {e}'; res = dict(s.counts)
    # 정지 구간에 분류 이벤트가 없어야 함
    in_stop, bad = False, []
    for line in s.events:
        if 'STOPPED' in line: in_stop = True
        elif 'RESTART' in line: in_stop = False
        elif in_stop and ('복귀 확인' in line or '직진 배출' in line): bad.append(line)
    checks = {
        '오류 없이 완료': error is None,
        '정지 1회 수행': res.get('STOP', 0) == 1,
        '재시작 1회 수행': res.get('RESTART', 0) == 1,
        '정지 중 분류 없음': not bad,
        '분류 1개 이상': res.get('DEFECT', 0) + res.get('BLUE', 0) + res.get('GREEN', 0) > 0,
    }
    with open('검증_결과.csv', 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.writer(f)
        w.writerow(['항목', '결과']); [w.writerow([k, 'PASS' if v else 'FAIL']) for k, v in checks.items()]
        w.writerow([]); w.writerow(['불량', res.get('DEFECT', 0)]); w.writerow(['정상파랑', res.get('BLUE', 0)]); w.writerow(['정상초록', res.get('GREEN', 0)])
        if error: w.writerow(['오류', error])
        w.writerow([]); w.writerow(['이벤트 로그']); [w.writerow([e]) for e in s.events]
    print('\n===== 검증 결과 =====')
    for k, v in checks.items(): print(('PASS ' if v else 'FAIL ') + k)
    print('수량:', {k: res.get(k, 0) for k in ('DEFECT', 'BLUE', 'GREEN')}, '| 오류:', error)
    return all(checks.values())

verify()


14:10:18 START  불량→P0 / 파랑→P1 / 초록→직진
14:10:22 VISION1 값=4
14:10:25 불량 P0 복귀 확인
14:10:28 VISION1 값=4
14:10:31 불량 P0 복귀 확인
14:10:34 VISION1 값=3
14:10:37 VISION2 값=3
14:10:38 STOP 요청 — 현재 동작 완료 후 정지
14:10:39 정상파랑 P1 복귀 확인
14:10:39 STOPPED (Start 버튼 또는 plan으로 재시작)
14:10:48 RESTART
14:10:52 VISION1 값=3
14:10:54 VISION2 값=3
14:10:57 정상파랑 P1 복귀 확인
14:10:59 VISION1 값=3
14:11:02 VISION2 값=3
14:11:04 정상파랑 P1 복귀 확인
14:11:08 VISION1 값=4
14:11:10 불량 P0 복귀 확인
14:11:14 VISION1 값=6
14:11:16 VISION2 값=6
14:11:16 정상초록 → 직진 배출
14:11:18 END {'DEFECT': 3, 'BLUE': 3, 'STOP': 1, 'RESTART': 1, 'GREEN': 1} state=FEED

===== 검증 결과 =====
PASS 오류 없이 완료
PASS 정지 1회 수행
PASS 재시작 1회 수행
PASS 정지 중 분류 없음
PASS 분류 1개 이상
수량: {'DEFECT': 3, 'BLUE': 3, 'GREEN': 1} | 오류: None


True